# Early classification pada light curve dengan fitur raw

Notebook ini menguji kemampuan model memprediksi kelas dari prefix kronologis light curve menggunakan enam fitur raw. Model tidak dilatih ulang: checkpoint seed 42 digunakan untuk menguji 6, 10, 15, 20, 30, dan seluruh detection. Pemotongan dilakukan berdasarkan jumlah detection riil kronologis dan detection setelah prefix dikeluarkan dari mask.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from alerce_classifier.training.loop import (
    HybridCNNGRU, CNNClassifier, GRUClassifier,
    evaluate_detection_prefixes, load_split, make_loader, evaluate,
)

DATA_DIR = Path('data/alerce_ztf_lc_sanchezsaez_full/data/full')
REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / 'pyproject.toml').is_file():
    REPO_ROOT = REPO_ROOT.parents[1]
RAW_OUTPUT_DIR = DATA_DIR / 'preprocessing' / 'notebook_output'
ARTIFACT_DIR = REPO_ROOT / 'artifacts'
OUTPUT_DIR = ARTIFACT_DIR / 'benchmark'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FEATURE_CONFIG = json.loads((ARTIFACT_DIR / 'feature_config.json').read_text(encoding='utf-8'))
FEATURE_COUNT = len(FEATURE_CONFIG['features'])
SEED = 42
BATCH_SIZE = 64
DETECTION_COUNTS = [6, 10, 15, 20, 30]
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
print('Feature count:', FEATURE_COUNT)
print('Detection counts:', DETECTION_COUNTS)


## 1. Validasi artifact dan memuat test sequence

In [ ]:
required = [
    RAW_OUTPUT_DIR / 'test_padded.npz',
    ARTIFACT_DIR / 'hybrid' / 'hybrid_cnn_gru_seed42_best.pt',
    ARTIFACT_DIR / 'cnn' / 'cnn_seed42_best.pt',
    ARTIFACT_DIR / 'gru' / 'gru_seed42_best.pt',
]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError(f'Artifact early classification tidak lengkap: {missing}')
test_tensors = load_split(RAW_OUTPUT_DIR / 'test_padded.npz')
test_x, test_mask, test_y = test_tensors
print('Test tensor:', tuple(test_x.shape))
print('Test labels:', torch.bincount(test_y, minlength=3).tolist())
print('Mean real sequence length:', float(test_mask.sum(dim=1).float().mean()))


## 2. Memuat checkpoint terbaik

Checkpoint dipilih pada eksperimen training berdasarkan validation macro-F1. Test set hanya digunakan untuk evaluasi prefix pada notebook ini.

In [ ]:
model_factories = {
    'hybrid_cnn_gru': HybridCNNGRU,
    'cnn': CNNClassifier,
    'gru': GRUClassifier,
}
models = {}
checkpoint_summary = {}
for model_name, factory in model_factories.items():
    model = factory(feature_count=FEATURE_COUNT).to(DEVICE)
    checkpoint_path = ARTIFACT_DIR / model_name / f'{model_name}_seed{SEED}_best.pt'
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    models[model_name] = model
    checkpoint_summary[model_name] = {
        'best_epoch': checkpoint['best_epoch'],
        'best_validation_macro_f1': checkpoint['best_validation_macro_f1'],
    }
print(json.dumps(checkpoint_summary, indent=2))


## 3. Evaluasi prefix kronologis

Untuk setiap OID dan fraction, hanya detection pertama sampai batas prefix yang dipertahankan. Nilai setelah batas tersebut menjadi nol dan mask-nya menjadi 0. Tidak ada retraining atau penggunaan detection masa depan.

In [ ]:
prefix_results = {}
for model_name, model in models.items():
    print(f'Evaluating {model_name}', flush=True)
    prefix_results[model_name] = evaluate_detection_prefixes(
        model, test_x, test_mask, test_y,
        [*DETECTION_COUNTS, None], DEVICE
    )
    for detection_count, metrics in prefix_results[model_name].items():
        print(
            f'  {detection_count}: accuracy={metrics["accuracy"]:.4f}, '
            f'balanced_accuracy={metrics["balanced_accuracy"]:.4f}, '
            f'macro_f1={metrics["macro_f1"]:.4f}'
        )
(OUTPUT_DIR / 'early_classification_metrics.json').write_text(
    json.dumps(prefix_results, indent=2), encoding='utf-8'
)


## 4. Ringkasan perbandingan model

In [ ]:
rows = []
for model_name, results in prefix_results.items():
    for detection_count, metrics in results.items():
        rows.append({
            'model': model_name,
            'detection_count': detection_count,
            'accuracy': metrics['accuracy'],
            'balanced_accuracy': metrics['balanced_accuracy'],
            'macro_f1': metrics['macro_f1'],
        })
summary = pd.DataFrame(rows)
summary['sort_count'] = summary['detection_count'].replace({'all': 999999}).astype(int)
summary = summary.sort_values(['sort_count', 'macro_f1'], ascending=[True, False])
display(summary)
summary.to_csv(OUTPUT_DIR / 'early_classification_summary.csv', index=False)


## 5. Grafik early classification

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for metric, axis in zip(('accuracy', 'balanced_accuracy', 'macro_f1'), axes):
    for model_name, group in summary.groupby('model'):
        group = group.sort_values('sort_count')
        axis.plot(group['sort_count'].replace({999999: 999}), group[metric] * 100, marker='o', label=model_name)
    axis.set_xlabel('Jumlah detection awal')
    axis.set_ylabel(f'{metric} (%)')
    axis.set_title(metric.replace('_', ' ').title())
    axis.set_xticks([6, 10, 15, 20, 30, 999])
    axis.grid(alpha=.3)
axes[0].legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'early_classification_curves.png', dpi=150)
plt.show()


## 6. Confusion matrix pada prefix awal

Confusion matrix berikut membantu melihat apakah keunggulan awal model berasal dari pengenalan kelas tertentu atau dari dominasi kelas Periodic.

In [ ]:
class_names = ['Periodic', 'Stochastic', 'Transient']
for detection_count in [*DETECTION_COUNTS, 'all']:
    key = str(fraction)
    fig, axes = plt.subplots(1, len(models), figsize=(15, 4))
    for axis, (model_name, results) in zip(axes, prefix_results.items()):
        sns.heatmap(
        results[str(detection_count)]['confusion_matrix'], annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names, ax=axis,
        )
        axis.set_title(f'{model_name} - {detection_count}')
        axis.set_xlabel('Predicted')
        axis.set_ylabel('Actual')
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / f"confusion_matrix_detection_{detection_count}.png", dpi=150)
    plt.show()


## Interpretasi yang diuji

Eksperimen menguji robustness model terhadap prefix dengan jumlah detection absolut 6, 10, 15, 20, dan 30, serta seluruh sequence. Model tidak dilatih ulang. Hasil ditafsirkan menggunakan Macro-F1, balanced accuracy, coverage, dan confusion matrix.